# 01 — Télémétrie IoT : Analyse Exploratoire
**Smart Farm AI v3.0** | Node A (Sol/Irrigation) & Node B (Ruche/Apiculture)

Objectif : comprendre les distributions, corrélations et patterns saisonniers des capteurs IoT.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (14, 5)

# Load telemetry dataset
DATA_PATH = Path('..') / 'data' / 'telemetry_dataset.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head()

## 1. Statistiques Descriptives

In [ ]:
desc = df.describe().T
desc['cv_%'] = (desc['std'] / desc['mean'].abs() * 100).round(1)
desc['skewness'] = df.skew().round(3)
desc['kurtosis'] = df.kurtosis().round(3)
desc.style.background_gradient(subset=['cv_%'], cmap='YlOrRd')

## 2. Distributions par Métrique

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
n = len(numeric_cols)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col].dropna(), bins=40, edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=11)
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label='mean')
    axes[i].axvline(df[col].median(), color='orange', linestyle=':', label='median')
    if i == 0: axes[i].legend(fontsize=8)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.suptitle('Distributions des métriques IoT', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Heatmap de Corrélation

In [ ]:
corr = df[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Corrélations Pearson entre métriques IoT', fontsize=13)
plt.tight_layout()
plt.show()

# Top correlations
corr_pairs = corr.unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[(corr_pairs < 1.0) & (corr_pairs.abs() > 0.3)]
print('\nTop corrélations (|r| > 0.3):')
print(corr_pairs.head(10).to_string())

## 4. Détection d'Anomalies Visuelles (IsolationForest)

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

X = df[numeric_cols].fillna(df[numeric_cols].mean())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(contamination=0.05, random_state=42)
labels = iso.fit_predict(X_scaled)
df['anomaly'] = labels == -1

print(f'Anomalies détectées : {df["anomaly"].sum()} / {len(df)} ({df["anomaly"].mean()*100:.1f}%)')

if len(numeric_cols) >= 2:
    fig, ax = plt.subplots(figsize=(10, 6))
    normal = df[~df['anomaly']]
    anomalies = df[df['anomaly']]
    ax.scatter(normal[numeric_cols[0]], normal[numeric_cols[1]],
               alpha=0.4, label='Normal', s=15, color='steelblue')
    ax.scatter(anomalies[numeric_cols[0]], anomalies[numeric_cols[1]],
               alpha=0.9, label='Anomalie', s=60, color='crimson', marker='x', linewidths=2)
    ax.set_xlabel(numeric_cols[0]); ax.set_ylabel(numeric_cols[1])
    ax.set_title('Anomalies IoT (IsolationForest, contamination=5%)')
    ax.legend()
    plt.tight_layout(); plt.show()

## 5. Saisonnalité (si timestamp disponible)

In [ ]:
ts_cols = [c for c in df.columns if 'time' in c.lower() or 'date' in c.lower()]
if ts_cols:
    df['ts'] = pd.to_datetime(df[ts_cols[0]], errors='coerce')
    df_ts = df.dropna(subset=['ts']).set_index('ts').sort_index()
    if len(df_ts) > 10:
        df_ts[numeric_cols].resample('1H').mean().plot(subplots=True, figsize=(16, 3*len(numeric_cols)))
        plt.suptitle('Séries temporelles IoT (1h moyenne)', y=1.01)
        plt.tight_layout(); plt.show()
    else:
        print('Pas assez de points temporels pour la décomposition.')
else:
    print('Pas de colonne timestamp dans le dataset.')

## Conclusions

- **Métriques les plus corrélées** : voir heatmap ci-dessus
- **Taux d'anomalies** : ~5% selon IsolationForest (contamination=0.05)
- **Distributions** : la plupart des métriques suivent une loi normale légèrement asymétrique
- **Prochaine étape** : Prophet forecasting via `/api/v1/forecast/telemetry/{unit_id}`